# Librerías

In [4]:
# Librerías
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# Módulos 
from astroquery.mpc import MPC
from math import ceil

# Lectura de archivos

In [5]:
nombre_cometa = r'C/ 2023 A3'

curva_de_luz_procesada_COBS_df = pd.read_csv(r'Bases_de_datos/curva_de_luz_procesada_COBS.txt')
curva_de_luz_interna_COBS_df = pd.read_csv(r'Bases_de_datos/curva_de_luz_interna_COBS.txt')
curva_de_luz_externa_COBS_df = pd.read_csv(r'Bases_de_datos/curva_de_luz_externa_COBS.txt')

curva_de_luz_procesada_MPC_df = pd.read_csv(r'Bases_de_datos/curva_de_luz_procesada_MPC.txt')
curva_de_luz_interna_MPC_df = pd.read_csv(r'Bases_de_datos/curva_de_luz_interna_MPC.txt')
curva_de_luz_externa_MPC_df = pd.read_csv(r'Bases_de_datos/curva_de_luz_externa_MPC.txt')


# Unión de las bases de datos

In [6]:
curva_de_luz_procesada_COBS_MPC_df = pd.concat([curva_de_luz_procesada_COBS_df, curva_de_luz_procesada_MPC_df])

curva_de_luz_procesada_COBS_MPC_df['obs_date'] = pd.to_datetime(curva_de_luz_procesada_COBS_MPC_df.obs_date) 

curva_de_luz_procesada_COBS_MPC_df.sort_values('obs_date', inplace=True)
curva_de_luz_procesada_COBS_MPC_df.reset_index(inplace = True, drop = True)

# curva_de_luz_procesada_COBS_MPC_df.head()

In [7]:
# curva_de_luz_procesada_COBS_MPC_df.info()

In [8]:
curva_de_luz_min_COBS_MPC_df = curva_de_luz_procesada_COBS_MPC_df.groupby(by = 'obs_date').min()
curva_de_luz_min_COBS_MPC_df.reset_index(inplace = True)
# curva_de_luz_min_COBS_MPC_df.head()

In [9]:
numero_elementos_grupo = 7

curva_de_luz_externa_COBS_MPC_df = curva_de_luz_min_COBS_MPC_df.copy()
curva_de_luz_externa_COBS_MPC_df['promedio_movil'] = curva_de_luz_externa_COBS_MPC_df.magnitud_reducida.rolling(window = numero_elementos_grupo, center= True).mean()
# curva_de_luz_externa_COBS_MPC_df.head()

In [10]:
curva_de_luz_max_COBS_MPC_df = curva_de_luz_procesada_COBS_MPC_df.groupby(by = 'obs_date').max()
curva_de_luz_max_COBS_MPC_df.reset_index(inplace = True)
# curva_de_luz_max_COBS_MPC_df.head()

In [11]:
numero_elementos_grupo = 7

curva_de_luz_interna_COBS_MPC_df = curva_de_luz_max_COBS_MPC_df.copy()
curva_de_luz_interna_COBS_MPC_df['promedio_movil'] = curva_de_luz_interna_COBS_MPC_df.magnitud_reducida.rolling(window = numero_elementos_grupo, center= True).mean()
# curva_de_luz_interna_COBS_MPC_df.head()

# Curvas de luz MPC_COBS (Bases de datos unidas) 

In [12]:
fig = go.Figure()

# CURVA DE LUZ REDUCIDA ---------------------------------------------------------------------------------------------- 

fig.add_trace(go.Scatter(x=curva_de_luz_procesada_COBS_MPC_df.delta_t, y=curva_de_luz_procesada_COBS_MPC_df.magnitud_reducida, 
mode='markers', name='magnitud reducida COBS-MPC',
marker=dict(
    color="#00CCFF",
    line=dict(width=1, color='DarkSlateGrey')
)))

# ENVOLVENTE ---------------------------------------------------------------------------------------------------

fig.add_trace(go.Scatter(x=curva_de_luz_externa_COBS_MPC_df.delta_t, y=curva_de_luz_externa_COBS_MPC_df.promedio_movil, 
mode='markers', name='Envolvente COBS-MPC',
marker=dict(
    color="#EBFC00",
    line=dict(width=1, color='DarkSlateGrey')
)))

# NUCLEO -------------------------------------------------------------------------------------------------------

fig.add_trace(go.Scatter(x=curva_de_luz_interna_COBS_MPC_df.delta_t, y=curva_de_luz_interna_COBS_MPC_df.promedio_movil, 
mode='markers', name='Envolvente COBS-MPC',
marker=dict(
    color="#FC0000",
    line=dict(width=1, color='DarkSlateGrey')
)))

# EJES Y TITULOS -----------------------------------------------------------------------------------------------

fig.update_layout(template='plotly_dark')
fig.update_yaxes(autorange="reversed")
fig.update_layout(template='plotly_dark', xaxis_title='t - Tq', yaxis_title='m(1,1,alpha)', 
    title = f'Max/Min Averaged Lightcurve of comet {nombre_cometa}')

fig.show()

# Curvas de luz individuales (Bases de datos separadas)

In [13]:
# Gráfica de luz promediada
fig = go.Figure()

# CURVA DE LUZ REDUCIDA ---------------------------------------------------------------------------------------------- 

fig.add_trace(go.Scatter(x=curva_de_luz_procesada_COBS_df.delta_t, y=curva_de_luz_procesada_COBS_df.magnitud_reducida, 
mode='markers', name='magnitud reducida COBS',
marker=dict(
    color="#00CCFF",
    line=dict(width=1, color='DarkSlateGrey')
)))

fig.add_trace(go.Scatter(x=curva_de_luz_procesada_MPC_df.delta_t, y=curva_de_luz_procesada_MPC_df.magnitud_reducida, 
mode='markers', name='magnitud reducida MPC', 
marker=dict(
    color="#B300FF",
    line=dict(width=1, color='DarkSlateGrey')
)))

# ENVOLVENTE ---------------------------------------------------------------------------------------------------

fig.add_trace(go.Scatter(x=curva_de_luz_externa_COBS_df.delta_t, y=curva_de_luz_externa_COBS_df.promedio_movil, 
mode='markers', name='Envolvente COBS',
marker=dict(
    color="#EBFC00",
    line=dict(width=1, color='DarkSlateGrey')
)))

fig.add_trace(go.Scatter(x=curva_de_luz_externa_MPC_df.delta_t, y=curva_de_luz_externa_MPC_df.promedio_movil, 
mode='markers', name='Envolvente MPC',
marker=dict(
    color="#FCA400",
    line=dict(width=1, color='DarkSlateGrey')
)))

# NUCLEO -------------------------------------------------------------------------------------------------------

fig.add_trace(go.Scatter(x=curva_de_luz_interna_COBS_df.delta_t, y=curva_de_luz_interna_COBS_df.promedio_movil, 
mode='markers', name='Nucleo COBS',
marker=dict(
    color="#FF0000",
    line=dict(width=1, color='DarkSlateGrey')
)))

fig.add_trace(go.Scatter(x=curva_de_luz_interna_MPC_df.delta_t, y=curva_de_luz_interna_MPC_df.promedio_movil, 
mode='markers', name='Nucleo MPC',
marker=dict(
    color="#FF006A",
    line=dict(width=1, color='DarkSlateGrey')
)))

# EJES Y TITULOS -------------------------------------------------------------------------------------------------

fig.update_layout(template='plotly_dark')
fig.update_yaxes(autorange="reversed")
fig.update_layout(template='plotly_dark', xaxis_title='t - Tq', yaxis_title='m(1,1,alpha)', 
    title = f'Max/Min Averaged Lightcurve of comet {nombre_cometa}')

fig.show()